[JobSpy Library](https://github.com/speedyapply/JobSpy)
proxy list

In [1]:
#%pip install -U python-jobspy
#%pip install ipyleaflet # python package for interactive maps in jupyter notebooks
#%pip install xyzservices # for basemaps

import csv
from jobspy import scrape_jobs
from ipyleaflet import Map, Marker, basemaps, MarkerCluster, Popup
from ipywidgets import HTML
import pandas as pd

#import xyzservices.providers as xyz # check if this is needed

In [2]:
def gen_linkedin_str(excludeTerms,excludeCompanies):
    """
    Generate a job search string with LinkedIn boolean search formatting
    """
    linkedinStr = "\"engineer\""
    indeedStr = "title:engineer"

    if excludeTerms: #add all of the search terms to exlude
        linkedinStr += " NOT ("
        indeedStr += " -title:("
        for term in excludeTerms:
            linkedinStr += f"\"{term}\" OR "
            if len(term.split(" ")) > 1:
                indeedStr += f"\"{term}\" OR "
            else:
                indeedStr += f"{term} OR "
        linkedinStr = linkedinStr[:-4]
        indeedStr = indeedStr[:-4]    
        linkedinStr += ")"
        indeedStr += ")"
        
    if excludeCompanies:
        linkedinStr += " NOT ("
        indeedStr += " -company:("
        for company in excludeCompanies:
            linkedinStr += f"\"{company}\" OR "
            if len(company.split(" ")) > 1:
                indeedStr += f"\"{company}\" OR "
            else:
                indeedStr += f"{company} OR "
        linkedinStr = linkedinStr[:-4]    
        indeedStr = indeedStr[:-4]    
        linkedinStr += ")"
        indeedStr += ")"

    return (linkedinStr, indeedStr)

In [11]:
# CONTROL PANEL
#IO Controls
saveCSV = False
fileName = "250814_1000jobs.csv"
filePath = "C:/users/broge/Desktop/"

#Job scraper controls
sitesToSearch = ["linkedin","indeed"] #"linkedin", "zip_recruiter", "google", "glassdoor", "bayt", "naukri", "bdjobs"
#searchStr =
locationToSearch = "Seattle, WA"
distanceToSearch = 25
numResults = 200 #number of postings to scrape. Roughly 52s/100jobs
postingAge = 72 #max posting age in hours

#Search String controls
excludeTerms = ["compute", "electrical", "plumbing", "front end", "frontend", "sales", "software", "full stack", "customer support",
                "integration", "civil", "backend", "devops", "machine learning", "geotechnical", "network", "cloud", "aerospace",
                "Customer Service", "rf testing", "data engineer", "avionics", "gpu", "ux engineer", "security", "salesforce",
                "compiler", "business intelligence", "principal", "it application", "transportation engineer", "ai/ml", "propulsion",
                "customer experience", "pcba", "cryptography", "intern"]
excludeCompanies = ["Anduril","General Dynamics Ordnance and Tactical Systems", "Palantir"]
(linkedinStr,indeedStr) = gen_linkedin_str(excludeTerms,excludeCompanies)

#Map controls
center = (47.6,-122.3) #(38,-95) = center of US
zoomLevel = 8 #4 is good to fit whole US
myBasemap = basemaps.OpenTopoMap #This is the type of map eg street, topographical, nighttime, etc. other options:basemaps.OpenStreetMap.Mapnik, basemaps.NASAGIBS.ViirsEarthAtNight2012

In [13]:
print(linkedinStr)
print(indeedStr)

"engineer" NOT ("compute" OR "electrical" OR "plumbing" OR "front end" OR "frontend" OR "sales" OR "software" OR "full stack" OR "customer support" OR "integration" OR "civil" OR "backend" OR "devops" OR "machine learning" OR "geotechnical" OR "network" OR "cloud" OR "aerospace" OR "Customer Service" OR "rf testing" OR "data engineer" OR "avionics" OR "gpu" OR "ux engineer" OR "security" OR "salesforce" OR "compiler" OR "business intelligence" OR "principal" OR "it application" OR "transportation engineer" OR "ai/ml" OR "propulsion" OR "customer experience" OR "pcba" OR "cryptography" OR "intern") NOT ("Anduril" OR "General Dynamics Ordnance and Tactical Systems" OR "Palantir")
title:engineer -title:(compute OR electrical OR plumbing OR "front end" OR frontend OR sales OR software OR "full stack" OR "customer support" OR integration OR civil OR backend OR devops OR "machine learning" OR geotechnical OR network OR cloud OR aerospace OR "Customer Service" OR "rf testing" OR "data enginee

[Tips on indeed boolean](https://www.reddit.com/r/jobs/comments/a8b3fj/my_tips_on_how_to_use_boolean_operations_on/)

In [14]:
#initialize empty dataframe
jobs = pd.DataFrame()

jobs = scrape_jobs(
    site_name = "indeed",
    search_term = indeedStr,
    location= locationToSearch,
    distance=distanceToSearch,
    results_wanted= numResults,
    hours_old=postingAge,
    country_indeed="USA"
    #proxies=["5.10.246.207:80","localhost"]
)
print(f"Found {len(jobs)} jobs")

if saveCSV:
    try: 
        jobs.to_csv(f"{filePath}{fileName}", mode='x', quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
        print(f"File '{filePath}{fileName}' created successfully.")
    except FileExistsError:
        print(f"File '{filePath}{fileName}' already exists. Overwriting prevented.")

Found 149 jobs


In [15]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(jobs)

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,in-e573e32c93f80705,indeed,https://www.indeed.com/viewjob?jk=e573e32c93f80705,http://www.indeed.com/job/it-engineer-e573e32c93f80705,IT Engineer,Electric Mirror,"Everett, WA, US",2025-09-29,fulltime,direct_data,...,Everett,201 to 500,$25M to $100M (USD),Electric Mirror is the global leader in Lighted Mirrors and Mirror TV Technology™.,None,None,None,None,None,None
1,in-a18561aeac67fd99,indeed,https://www.indeed.com/viewjob?jk=a18561aeac67fd99,https://www1.jobdiva.com/candidates/myjobs/openjob_outside.jsp?a=ndjdnw7vk4y5q8swci59eedy2w472z05039bawkobc550aecw9ph653e8wk1g6bh&SearchString=&StatesString=&source=indeed.com&id=31225269&compid=-1,Manufacturing Quality Engineer,Katalyst Healthcares & Life Sciences,"Bothell, WA, US",2025-09-29,contract,NaN,...,"285 Durham Ave, Suite 12, South Plainfield, NJ 07080 United States.",51 to 200,$5M to $25M (USD),Katalyst HLS is a Global Contract Clinical Research Organisation that provides End-to-End Services for Phase I-IV.,None,None,None,None,None,None
2,in-11074a1002cad2d3,indeed,https://www.indeed.com/viewjob?jk=11074a1002cad2d3,https://jobs.pse.com/job/Snoqualmie-Senior-Engineer-Substation-Physical-Design-WA-98065/1312280200/?feedId=291300&utm_source=Indeed&utm_campaign=PSE_Indeed,Senior Engineer - Substation Physical Design,Puget Sound Energy,"Snoqualmie, WA, US",2025-09-28,NaN,direct_data,...,"Bellevue, WA","1,001 to 5,000",Decline to state,Puget Sound Energy is a Washington state energy utility providing electrical power and natural gas primarily in the Puget Sound region of the northwest United States.,None,None,None,None,None,None
3,in-8e11d63f848cbecf,indeed,https://www.indeed.com/viewjob?jk=8e11d63f848cbecf,https://jobs.pse.com/job/Snoqualmie-Engineer-III-Substation-Physical-Design-WA-98065/1312280100/?feedId=291300&utm_source=Indeed&utm_campaign=PSE_Indeed,Engineer III - Substation Physical Design,Puget Sound Energy,"Snoqualmie, WA, US",2025-09-28,NaN,direct_data,...,"Bellevue, WA","1,001 to 5,000",Decline to state,Puget Sound Energy is a Washington state energy utility providing electrical power and natural gas primarily in the Puget Sound region of the northwest United States.,None,None,None,None,None,None
4,in-2e418a2b8064cb90,indeed,https://www.indeed.com/viewjob?jk=2e418a2b8064cb90,https://jsv3.recruitics.com/redirect?rx_cid=3427&rx_jobId=200623047-3337&rx_url=https%3A%2F%2Fjobs.apple.com%2Fen-us%2Fdetails%2F200623047-3337%2Fsenior-site-reliability-engineer%3Fboard_id%3DJB001%26rx_campaign%3Dindeed0%26rx_ch%3Djobp4p%26rx_group%3D130780%26rx_id%3D7b469956-9c26-11f0-ba4b-6911854b0b11%26rx_job%3D200623047-3337%26rx_medium%3Dcpc%26rx_r%3Dnone%26rx_source%3Dindeed%26rx_ts%3D20250929T103802Z%26rx_vp%3Dcpc%26team%3DSFTWR,Senior Site Reliability Engineer,Apple,"Seattle, WA, US",2025-09-27,NaN,direct_data,...,"Cupertino, CA","10,000+",more than $10B (USD),This is where you can do the best work of your life.,None,None,None,None,None,None
5,in-97517e7e183da66f,indeed,https://www.indeed.com/viewjob?jk=97517e7e183da66f,https://asrinternationalcorp.applytojob.com/apply/GnwewXo9mw/Construction-Management-SupportField-Engineer?source=INDE&~,Construction Management Support/Field Engineer,ASR International,"Bremerton, WA, US",2025-09-27,fulltime,direct_data,...,"Hauppauge, NY",201 to 500,$5M to $25M (USD),NaN,None,None,None,None,None,None
6,in-4df2e5cce37ab77e,indeed,https://www.indeed.com/viewjob?jk=4df2e5cce37ab77e,https://gly.bamboohr.com/careers/59?source=indeed&src=indeed&postedDate=2025-09-27,Senior Project Engineer,GLY Construction,"Bellevue, WA, US",2025-09-27,NaN,direct_data,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
7,in-400c8e1c88feada1,indeed,https://www.indeed.com/viewjob?jk=400c8e1c88feada1,https://jobs.pse.com/

In [125]:
def get_coords(cityStateStr):
    """
    This function retrieves the latitude and longitude of a city.
    The job scraper returns a location for each job as a string with the format "city, SS" where SS is a two letter state abbrev.
    The state and city are looked up in a table of geographic data of US cities.
    """
    
    cityStateStr = cityStateStr.replace(", US","") #Indeed results are formatted City, SS, US. Remove the ", US" to make formatting consistent
    [city,state] = cityStateStr.split(", ")

    file_path = r"C:\Users\broge\Documents\GitHub\job-map\uscities.csv"
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if city == row["city"] and state == row["state_id"]:
                return float(row["lat"]), float(row["lng"])
    return ""

#if a jobs dataframe doesn't exist (ie the scraper wasn't just used) assume the user already has a scraped csv
if len(jobs)==0:
    jobs = pd.read_csv(f"{filePath}{fileName}")
    jobs = jobs.fillna('') #if importing a csv, convet any NaN floats to empty strings

#To plot a marker for each job on a map, we want job title, url, company, and location (latitude+longitude)
markerList = []
for i in range(len(jobs.location)):
    #extract relevant info for each entry of jobs dataframe
    locStr = jobs.location[i]
    url = jobs.job_url[i]
    title = jobs.title[i]
    company = jobs.company[i]

    #check if the entry has a location listed; if so then look up the lat,long of the city
    if locStr:
        coords = get_coords(locStr)
        currentMarker = Marker(location=coords,draggable=False)
        currentMessage = HTML()
        currentMessage.value = f"<a href=\"{url}\">{title}</a><br>{company}"
        currentMarker.popup = currentMessage

        markerList.append(currentMarker)

#Generate map with list of markers from above created above
map = Map(basemap=myBasemap, center=center, zoom=zoomLevel)
cluster = MarkerCluster(markers=markerList,max_cluster_radius=1) #default radius = 80 pixels
map.add(cluster)

Map(center=[47.6, -122.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

In [ ]:
print(indeedStr)



title:engineer -title:(compute OR electrical OR plumbing OR "front end" OR frontend OR sales OR software OR "full stack" OR "customer support" OR integration OR civil OR backend OR devops OR "machine learning" OR geotechnical OR network OR cloud OR aerospace) -company:(Anduril)


In [90]:
for item in jobs.title:
    print(item)

Sourcing Engineer II
Sourcing Engineer II
Lead Transportation Engineer / Project Manager – Roads + Highways
Technical Support Engineer
Radar System Engineer (Expert)
Associate or Experienced Quality Engineer
Detection Engineer
Mechanical Hydraulics Design & Analysis Engineer (Associate or Experienced)
Product Review Engineer (Liaison Engineering)
AI/ML - Applied Research Engineer, Machine Translation
Telecom/EV Design Engineer
Mechanical Project Engineer
Mechanical Project Engineer
Principal Customer Experience Engineer
Systems Engineer I
Founding Engineer
Field Engineer - Kitsap WA
Mechanical/Fluids Design Engineer II - Test Development
Propulsion Engineer III - Thruster Design & Development
Principal Customer Experience Engineer
Hardware Engineer
ASIC Design Verification Engineer, Kuiper Modem DV Team
Regional Environmental Engineer, Air Compliance, AWS Environmental Team
Propulsion Flight Operations Engineer, Project Kuiper
Opto-Mechanical Engineer, Reality Labs Research, Hardware E